## Layers and Modules

In [3]:
import torch
from torch import nn
from torch.nn import functional as F

In [4]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
X = torch.rand(2, 20)
print("X:", X, '\n')
print("X shape:", X.shape)
net(X).shape

X: tensor([[0.0313, 0.0322, 0.7678, 0.9669, 0.4450, 0.9099, 0.4410, 0.9229, 0.4428,
         0.5110, 0.7665, 0.0118, 0.5002, 0.0134, 0.0703, 0.4312, 0.8705, 0.6010,
         0.0548, 0.9890],
        [0.1378, 0.8157, 0.8858, 0.5548, 0.9772, 0.5779, 0.2265, 0.7671, 0.3447,
         0.5995, 0.8547, 0.8810, 0.7913, 0.4169, 0.5771, 0.6206, 0.0595, 0.3992,
         0.0698, 0.5242]]) 

X shape: torch.Size([2, 20])


C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


torch.Size([2, 10])

### A Custom Module

In [6]:
class MLP(nn.Module):
    def __init__(self):
        # Call the constructor of the parent class nn.Module to perform
        # the necessary initialization
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    # Define the forward propagation of the model, that is, how to return the
    # required model output based on the input X
    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [17]:
net = MLP()
net(X).shape

torch.Size([2, 10])

### The Seuquential Module

In [10]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self.add_module(str(idx), module)

    def forward(self, X):
        for module in self.children():
            X = module(X)
        return X

In [12]:
net = MySequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
net(X).shape

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


torch.Size([2, 10])

In [13]:
net.forward(X).shape

torch.Size([2, 10])

### Executing Code in the Forward Propagation Method

In [18]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20))
        self.linear = nn.LazyLinear(20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(X @ self.rand_weight + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [19]:
torch.rand((20, 20))

tensor([[0.0166, 0.4372, 0.5132, 0.1734, 0.3063, 0.9950, 0.2662, 0.4391, 0.7557,
         0.7498, 0.8998, 0.8379, 0.2923, 0.9132, 0.8342, 0.3845, 0.4481, 0.4961,
         0.3443, 0.9640],
        [0.6481, 0.4661, 0.3755, 0.3088, 0.5662, 0.3794, 0.1674, 0.9424, 0.9103,
         0.5455, 0.2183, 0.0088, 0.5343, 0.1152, 0.4559, 0.6293, 0.2652, 0.8338,
         0.1890, 0.2608],
        [0.8285, 0.0140, 0.1488, 0.8225, 0.5809, 0.7151, 0.6083, 0.3561, 0.4208,
         0.0920, 0.8746, 0.0655, 0.5104, 0.0900, 0.8697, 0.5694, 0.5615, 0.2056,
         0.3333, 0.9615],
        [0.5365, 0.6949, 0.7430, 0.7018, 0.0845, 0.9510, 0.0342, 0.1602, 0.6203,
         0.3058, 0.8696, 0.6243, 0.1003, 0.8239, 0.4781, 0.0137, 0.2969, 0.2230,
         0.9199, 0.8172],
        [0.7992, 0.7130, 0.5263, 0.4011, 0.7193, 0.9396, 0.2387, 0.0820, 0.2310,
         0.2311, 0.2671, 0.3389, 0.4466, 0.8522, 0.0942, 0.7159, 0.3114, 0.2699,
         0.1162, 0.1545],
        [0.9354, 0.7490, 0.2715, 0.8650, 0.2919, 0.6363, 0.4

In [20]:
net = FixedHiddenMLP()
net(X)

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


tensor(-0.0175, grad_fn=<SumBackward0>)

In [21]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.LazyLinear(64), nn.ReLU(),
                                 nn.LazyLinear(32), nn.ReLU())
        self.linear = nn.LazyLinear(16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.LazyLinear(20), FixedHiddenMLP())
chimera(X)

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


tensor(0.0031, grad_fn=<SumBackward0>)

## Exercises
####

### Ex. 2

In [23]:
X = torch.rand(2, 20)

In [22]:
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [27]:
class ParallelModule(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1 = net1
        self.net2 = net2

    def forward(self, X):
        return torch.cat((self.net1(X), self.net2(X)))        

In [28]:
Parall = ParallelModule(MLP(), MLP2())
output = Parall(X)

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [30]:
output, output.shape

(tensor([[ 0.2299,  0.0430, -0.2310,  0.0092,  0.0240,  0.0918,  0.1487, -0.2502,
           0.0763, -0.1728],
         [ 0.3017,  0.1453, -0.2779,  0.1286, -0.0531,  0.0659,  0.1677, -0.2500,
           0.0925, -0.1293],
         [-0.0871, -0.1676, -0.2604, -0.1240, -0.0035,  0.0179,  0.0593, -0.0874,
          -0.4598,  0.0459],
         [-0.1511, -0.1839, -0.3435, -0.1416, -0.0354, -0.0565, -0.0369, -0.0856,
          -0.4615, -0.0472]], grad_fn=<CatBackward0>),
 torch.Size([4, 10]))

### Ex. 3

In [38]:
class ConcatNet(nn.Module):
    def __init__(self, mlp_class, k):
        super().__init__()
        self.mlps = nn.ModuleList([mlp_class() for _ in range(k)])

    def forward(self, X):
        outputs = [net(X) for net in self.mlps]
        return torch.cat(outputs, dim=1)

In [39]:
cnct_net = ConcatNet(MLP2, 10)
outpts = cnct_net(X)

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [40]:
outpts, outpts.shape

(tensor([[-7.6658e-02, -8.2790e-03, -1.2737e-01,  1.2661e-01,  8.2062e-02,
           7.0246e-02,  8.4409e-02, -2.5342e-02, -1.5679e-01,  1.9942e-01,
          -1.2169e-01,  1.7538e-01,  1.1753e-02, -6.4781e-02, -1.7485e-01,
          -1.6810e-02,  5.7015e-02,  9.4014e-02, -4.0628e-03, -1.8475e-02,
           1.3454e-01, -9.1631e-02, -2.0240e-01,  1.0169e-01,  2.5270e-01,
           7.9751e-02, -4.8060e-02, -6.6687e-02,  1.3927e-02, -1.8872e-01,
           2.0111e-01,  4.2557e-02,  9.9117e-02,  2.0776e-01, -1.8337e-02,
          -1.5606e-01,  7.3339e-02,  2.3143e-01, -1.5107e-01,  8.7182e-02,
          -6.8485e-02,  7.8385e-03,  2.4365e-01, -2.1613e-01, -4.4671e-02,
          -5.8824e-02,  7.1973e-04,  3.0217e-02, -2.6169e-02, -3.4143e-02,
          -7.2575e-02,  2.7589e-01,  8.7962e-03,  1.0370e-01, -1.8960e-01,
          -1.1693e-01, -1.2021e-01,  2.9747e-01,  2.6406e-01, -7.8274e-02,
           7.4339e-02, -8.0447e-02, -1.3602e-01, -5.2254e-02,  1.8445e-01,
          -9.2152e-02, -2